# Quantitative Contrarian Trading & Behavioural Research

# Setup

## Imports

In [ ]:
from pathlib import Path
import json
import re
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

np.random.seed(42)

print("Imports loaded.")

Imports loaded.


## Paths

In [ ]:
def find_project_root():
    current=Path.cwd().resolve()
    for candidate in [current,*current.parents]:
        if (candidate/"requirements.txt").exists() and (candidate/"notebooks").exists():
            return candidate
    return current.parent if current.name=="notebooks" else current

PROJECT_ROOT=find_project_root()
HISTORICAL_TRADES_DIR=PROJECT_ROOT/"data"/"User Trades"
UNSEEN_TRADES_DIR=PROJECT_ROOT/"data"/"Unseen User Trades"
MODELS_DIR=PROJECT_ROOT/"models"
MODEL_PATH=MODELS_DIR/"exp10b_model.joblib"
MODEL_METADATA_PATH=MODELS_DIR/"exp10b_model_metadata.json"
OUTPUT_DIR=PROJECT_ROOT/"outputs"/"inference"
DECISIONS_OUTPUT_PATH=OUTPUT_DIR/"unseen_trade_decisions.csv"
EVALUATION_SUMMARY_PATH=OUTPUT_DIR/"evaluation_summary.csv"
CAMPAIGN_PERFORMANCE_PATH=OUTPUT_DIR/"campaign_performance.csv"
BOOTSTRAP_SAMPLES_PATH=OUTPUT_DIR/"bootstrap_samples.csv"

print("Project root:",PROJECT_ROOT)
print("Historical data:",HISTORICAL_TRADES_DIR)
print("Unseen data:",UNSEEN_TRADES_DIR)
print("Output directory:",OUTPUT_DIR)

Project root: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design
Historical data: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/User Trades
Unseen data: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/Unseen User Trades
Output directory: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/inference


## Constants

In [ ]:
RANDOM_SEED=42
STARTING_BALANCE=5000.0
MAX_DRAWDOWN_RATE=0.04
DRAWDOWN_LIMIT_AMOUNT=STARTING_BALANCE*MAX_DRAWDOWN_RATE
REALIZED_DRAWDOWN_BOUNDARY=-DRAWDOWN_LIMIT_AMOUNT
TRAINING_CAMPAIGN_MIN=33
TRAINING_CAMPAIGN_MAX=52
DECISION_THRESHOLD=85.102037
N_BOOTSTRAP=1000
CONFIDENCE_LEVEL=0.95
STAND_DOWN_FILTER_ENABLED=False

FEATURE_COLUMNS=[
"post_loss_entry","post_loss_reentry_gap_minutes","post_loss_amount_ratio",
"past_trade_open_count","elapsed_active_hours","past_trades_opened_per_active_hour",
"trades_opened_past_30_minutes","minutes_since_previous_trade_open",
"past_median_trade_open_gap_minutes","trade_open_gap_to_past_median_ratio",
"minutes_since_previous_idea_start","past_median_idea_start_gap_minutes",
"idea_start_gap_to_past_median_ratio","realized_distance_to_drawdown_limit",
"historical_median_amount_before_trade","current_to_historical_median_amount_ratio",
"past_amount_cv","past_completed_idea_count","past_idea_win_rate",
"past_mean_win_profit_per_lot","past_mean_loss_abs_profit_per_lot","past_payoff_ratio"]

LIGHTGBM_PARAMETERS={
    "n_estimators":200,
    "learning_rate":0.05,
    "num_leaves":15,
    "random_state":RANDOM_SEED,
    "verbosity":-1
}

print("Frozen feature count:",len(FEATURE_COLUMNS))
print("Frozen decision threshold:",DECISION_THRESHOLD)

Frozen feature count: 22
Frozen decision threshold: 85.102037


## Input checks

In [26]:
print("Checking required input directories...")
missing=[p for p in [HISTORICAL_TRADES_DIR,UNSEEN_TRADES_DIR] if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required directories:\n- "+"\n- ".join(map(str,missing)))
MODELS_DIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print("Input directories found.")

Checking required input directories...
Input directories found.


# Data preparation

## Raw trade schema

In [27]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)


TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]


TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]


TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]


TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price":
        "open_trade_cross_price",
    "opentradecrossprice":
        "open_trade_cross_price",

    "close_trade_cross_price":
        "close_trade_cross_price",
    "closetradecrossprice":
        "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}

## Column normalization

In [28]:
def normalize_column_name(
    column,
):
    """Convert a raw column name to lowercase snake case."""

    column_name = str(
        column
    ).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return (
        column_name
        .strip("_")
        .lower()
    )

## Campaign parsing

In [29]:
def parse_campaign_date(
    date_string,
):
    """Parse the campaign date encoded in a filename."""

    normalized_date_string = re.sub(
        r"[_\s]+",
        " ",
        date_string.strip(),
    )

    for date_format in [
        "%d %b %Y",
        "%d %B %Y",
    ]:
        parsed_date = pd.to_datetime(
            normalized_date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(
            parsed_date
        ):
            return parsed_date

    return pd.NaT

In [30]:
def parse_filename(
    path,
):
    """Extract campaign metadata from a raw trade filename."""

    path = Path(
        path
    )

    filename = (
        path.stem
    )

    match = (
        FILENAME_PATTERN
        .search(
            filename
        )
    )

    if match:
        return {
            "campaign_id":
                int(
                    match.group(1)
                ),
            "campaign_date":
                parse_campaign_date(
                    match.group(2)
                ),
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(
            campaign_match.group(1)
        )
        if campaign_match
        else None
    )

    return {
        "campaign_id":
            campaign_id,
        "campaign_date":
            pd.NaT,
    }

## Trade loading

In [31]:
def load_trade_file(
    path,
):
    """Load and standardize one raw campaign trade file."""

    path = Path(
        path
    )

    metadata = (
        parse_filename(
            path
        )
    )

    if (
        metadata[
            "campaign_id"
        ]
        is None
    ):
        raise ValueError(
            "Could not determine campaign ID "
            f"from {path.name}."
        )

    if (
        path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            path,
            low_memory=False,
        )

    elif (
        path.suffix.lower()
        == ".xlsx"
    ):
        df = pd.read_excel(
            path
        )

    else:
        raise ValueError(
            f"Unsupported file type: "
            f"{path.suffix}"
        )

    normalized_columns = {
        column:
            normalize_column_name(
                column
            )
        for column in df.columns
    }

    df = df.rename(
        columns=normalized_columns
    )

    df = df.rename(
        columns=TRADE_COLUMN_ALIASES
    )

    df.insert(
        0,
        "source_row_number",
        np.arange(
            2,
            len(df) + 2,
        ),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "close_date_time",
        "amount",
        "net_profit",
        "side",
    }

    missing_required = (
        required_columns
        - set(
            df.columns
        )
    )

    if missing_required:
        raise ValueError(
            f"{path.name} is missing "
            "required columns: "
            f"{sorted(missing_required)}"
        )

    header_echo_mask = (
        df[
            "account_id"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "account_id",
                "accountid",
                "account",
            }
        )
        .fillna(False)
        |
        df[
            "open_date_time"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "open_date_time",
                "opendatetime",
            }
        )
        .fillna(False)
    )

    df = (
        df.loc[
            ~header_echo_mask
        ]
        .copy()
    )

    for column in (
        TRADE_ID_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace(
                    {
                        "":
                            pd.NA,
                        "nan":
                            pd.NA,
                        "None":
                            pd.NA,
                    }
                )
            )

    for column in (
        TRADE_NUMERIC_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    for column in (
        TRADE_DATETIME_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_datetime(
                df[column],
                errors="coerce",
                utc=True,
            )

    df[
        "side"
    ] = (
        df[
            "side"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    if (
        "currency"
        in df.columns
    ):
        df[
            "currency"
        ] = (
            df[
                "currency"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df[
        "campaign_id"
    ] = metadata[
        "campaign_id"
    ]

    df[
        "campaign_date"
    ] = metadata[
        "campaign_date"
    ]

    df[
        "source_file"
    ] = path.name

    df[
        "source_path"
    ] = str(
        path
    )

    return (
        df.reset_index(
            drop=True
        )
    )

In [32]:
def load_trade_directory(
    directory,
    is_unseen,
):
    """Load every supported trade file in a directory."""

    directory = Path(
        directory
    )

    dataset_label = (
        "UNSEEN"
        if is_unseen
        else "HISTORICAL"
    )

    print(
        f"\nScanning {dataset_label.lower()} "
        f"trade directory:"
    )
    print(
        f"  {directory}"
    )

    if not directory.exists():
        raise FileNotFoundError(
            f"Trade directory not found: "
            f"{directory.resolve()}"
        )

    paths = sorted(
        [
            path
            for path in directory.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in {
                    ".csv",
                    ".xlsx",
                }
            )
        ]
    )

    if not paths:
        raise ValueError(
            "No CSV or XLSX trade files "
            f"found in {directory}."
        )

    print(
        f"Found {len(paths)} "
        f"{dataset_label.lower()} files."
    )

    frames = []

    for file_number, path in enumerate(
        paths,
        start=1,
    ):
        print(
            f"  [{file_number}/{len(paths)}] "
            f"Processing: {path.name}"
        )

        frame = (
            load_trade_file(
                path
            )
        )

        frame[
            "_is_unseen"
        ] = bool(
            is_unseen
        )

        frames.append(
            frame
        )

    combined = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Loaded {len(combined):,} "
        f"{dataset_label.lower()} trades."
    )

    return combined

## Chronological preprocessing

In [33]:
def attach_previous_completed_trade(
    trades,
):
    """Attach the most recent completed trade known at entry time."""

    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in (
        trades.groupby(
            grouping_columns,
            dropna=False,
            sort=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        completed = (
            group.loc[
                group[
                    "close_date_time"
                ].notna()
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
        ]

        if (
            "position_id"
            in completed.columns
        ):
            previous_columns.append(
                "position_id"
            )

        previous = (
            completed[
                previous_columns
            ]
            .rename(
                columns={
                    "close_date_time":
                        (
                            "previous_completed_"
                            "close_date_time"
                        ),
                    "net_profit":
                        (
                            "previous_completed_"
                            "net_profit"
                        ),
                    "amount":
                        (
                            "previous_completed_"
                            "amount"
                        ),
                    "position_id":
                        (
                            "previous_completed_"
                            "position_id"
                        ),
                }
            )
        )

        merged = pd.merge_asof(
            current.sort_values(
                "open_date_time"
            ),
            previous.sort_values(
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "previous_completed_"
                "close_date_time"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(
            merged
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "previous_completed_was_loss"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        < 0
    )

    result[
        "previous_completed_was_win"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        > 0
    )

    result[
        "reentry_gap_minutes"
    ] = (
        (
            result[
                "open_date_time"
            ]
            - result[
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ]
        )
        .dt.total_seconds()
        / 60
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [34]:
def assign_trade_ideas(
    trades,
    maximum_gap_minutes=3.0,
):
    """Assign the Stage 1 directional trade-idea identifier."""

    result = (
        trades.copy()
    )

    group_columns = [
        "account_id",
        "campaign_id",
        "side",
    ]

    result = (
        result
        .sort_values(
            group_columns
            + [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    idea_numbers = pd.Series(
        index=result.index,
        dtype="Int64",
    )

    for _, group in (
        result.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        group = group.sort_values(
            [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )

        current_idea_number = 1
        current_idea_latest_close = (
            pd.NaT
        )

        for row in (
            group.itertuples()
        ):
            if pd.isna(
                current_idea_latest_close
            ):
                current_idea_latest_close = (
                    row.close_date_time
                )

            else:
                gap_minutes = (
                    (
                        row.open_date_time
                        - current_idea_latest_close
                    )
                    .total_seconds()
                    / 60
                )

                if (
                    gap_minutes
                    > maximum_gap_minutes
                ):
                    current_idea_number += 1

                    current_idea_latest_close = (
                        row.close_date_time
                    )

                else:
                    current_idea_latest_close = max(
                        current_idea_latest_close,
                        row.close_date_time,
                    )

            idea_numbers.loc[
                row.Index
            ] = (
                current_idea_number
            )

    result[
        "idea_number"
    ] = (
        idea_numbers
    )

    result[
        "idea_id"
    ] = (
        result[
            "account_id"
        ].astype(str)
        + "_"
        + result[
            "campaign_id"
        ].astype(str)
        + "_"
        + result[
            "side"
        ].astype(str)
        + "_"
        + result[
            "idea_number"
        ].astype(str)
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [35]:
def build_stage1_inference_tables(
    historical_trades_dir,
    unseen_trades_dir,
):
    """Build the Stage 1 tables required by the frozen model.

    Historical and unseen trades are processed together so that
    pre-trade historical features for unseen trades can use information
    that would already have been available before the unseen trade opened.

    Args:
        historical_trades_dir: Directory containing known historical trades.
        unseen_trades_dir: Directory containing the hidden/unseen trades.

    Returns:
        Dictionary containing standardized trade tables and idea-level data.
    """

    historical_trades = (
        load_trade_directory(
            directory=(
                historical_trades_dir
            ),
            is_unseen=False,
        )
    )

    unseen_trades = (
        load_trade_directory(
            directory=(
                unseen_trades_dir
            ),
            is_unseen=True,
        )
    )

    trades = pd.concat(
        [
            historical_trades,
            unseen_trades,
        ],
        ignore_index=True,
    )

    trades = (
        trades
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "source_row_number",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    trades[
        "trade_row_id"
    ] = np.arange(
        1,
        len(trades) + 1,
    )

    trades_with_previous_completed = (
        attach_previous_completed_trade(
            trades
        )
    )

    trades_with_ideas = (
        assign_trade_ideas(
            trades=trades,
            maximum_gap_minutes=3.0,
        )
    )

    trades_with_ideas = (
        trades_with_ideas
        .sort_values(
            [
                "idea_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    trades_with_ideas[
        "trade_number_within_idea"
    ] = (
        trades_with_ideas
        .groupby(
            "idea_id"
        )
        .cumcount()
        + 1
    )

    idea_features = (
        trades_with_ideas
        .groupby(
            "idea_id",
            as_index=False,
        )
        .agg(
            account_id=(
                "account_id",
                "first",
            ),
            campaign_id=(
                "campaign_id",
                "first",
            ),
            side=(
                "side",
                "first",
            ),
            idea_start_time=(
                "open_date_time",
                "min",
            ),
            idea_end_time=(
                "close_date_time",
                "max",
            ),
            total_amount=(
                "amount",
                "sum",
            ),
            total_net_profit=(
                "net_profit",
                "sum",
            ),
        )
    )

    idea_features[
        "is_profitable_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        > 0
    )

    idea_features[
        "is_losing_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        < 0
    )

    idea_features[
        "is_breakeven_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        == 0
    )

    unseen_trade_row_ids = (
        trades.loc[
            trades[
                "_is_unseen"
            ],
            "trade_row_id",
        ]
        .tolist()
    )

    return {
        "trades":
            trades,
        "trades_with_previous_completed":
            (
                trades_with_previous_completed
            ),
        "trades_with_ideas":
            trades_with_ideas,
        "idea_features":
            idea_features,
        "unseen_trade_row_ids":
            unseen_trade_row_ids,
    }

# Feature engineering

## Historical activity

In [36]:
def attach_past_event_count(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    output_column,
):
    """Attach the number of strictly earlier events."""

    event_counts = (
        event_table
        .groupby(
            group_columns
            + [
                event_time_column
            ],
            dropna=False,
        )
        .size()
        .rename(
            "_events_at_timestamp"
        )
        .reset_index()
    )

    event_counts = (
        event_counts
        .sort_values(
            group_columns
            + [
                event_time_column
            ]
        )
    )

    event_counts[
        output_column
    ] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )[
            "_events_at_timestamp"
        ]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_counts.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_counts[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_counts[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(
                event_time_column
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                output_column
            ] = 0

            output_groups.append(
                current_group
            )

            continue

        merge_time_column = (
            f"_past_{output_column}_time"
        )

        group_events = (
            group_events.rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = (
            pd.merge_asof(
                left=(
                    current_group
                ),
                right=(
                    group_events
                ),
                left_on=(
                    target_time_column
                ),
                right_on=(
                    merge_time_column
                ),
                direction="backward",
                allow_exact_matches=False,
            )
        )

        current_group[
            output_column
        ] = (
            current_group[
                output_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

In [37]:
def attach_rolling_event_counts(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    window_minutes,
    feature_prefix,
):
    """Attach counts of strictly earlier events in rolling windows."""

    result_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        target_times = (
            current_group[
                target_time_column
            ]
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        event_times = (
            group_events
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        for window in (
            window_minutes
        ):
            left_boundaries = (
                target_times
                - np.timedelta64(
                    window,
                    "m",
                )
            )

            left_indices = (
                np.searchsorted(
                    event_times,
                    left_boundaries,
                    side="left",
                )
            )

            right_indices = (
                np.searchsorted(
                    event_times,
                    target_times,
                    side="left",
                )
            )

            feature_column = (
                f"{feature_prefix}"
                f"_past_{window}_minutes"
            )

            current_group[
                feature_column
            ] = (
                right_indices
                - left_indices
            )

        result_groups.append(
            current_group
        )

    return pd.concat(
        result_groups,
        ignore_index=True,
    )

## Trade timing

In [38]:
def attach_entry_spacing_features(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    tie_breaker_columns,
    previous_gap_column,
    past_median_gap_column,
    gap_ratio_column,
):
    """Attach pre-entry gap and historical median pace features."""

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask
            ]
            .sort_values(
                [
                    event_time_column
                ]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                previous_gap_column
            ] = np.nan

            current_group[
                past_median_gap_column
            ] = np.nan

            current_group[
                gap_ratio_column
            ] = np.nan

            output_groups.append(
                current_group
            )

            continue

        event_gap_column = (
            "_entry_gap_minutes"
        )

        group_events[
            event_gap_column
        ] = (
            group_events[
                event_time_column
            ]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[
            past_median_gap_column
        ] = (
            group_events[
                event_gap_column
            ]
            .expanding(
                min_periods=1
            )
            .median()
        )

        merge_time_column = (
            "_previous_event_time"
        )

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                events_for_merge
            ),
            left_on=(
                target_time_column
            ),
            right_on=(
                merge_time_column
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            previous_gap_column
        ] = (
            (
                current_group[
                    target_time_column
                ]
                - current_group[
                    merge_time_column
                ]
            )
            .dt.total_seconds()
            / 60
        )

        valid_ratio = (
            current_group[
                previous_gap_column
            ].ge(0)
            & current_group[
                past_median_gap_column
            ].gt(0)
        )

        current_group[
            gap_ratio_column
        ] = np.where(
            valid_ratio,
            (
                current_group[
                    previous_gap_column
                ]
                / current_group[
                    past_median_gap_column
                ]
            ),
            np.nan,
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

## Challenge state

In [39]:
def attach_realized_challenge_state(
    target_table,
    realized_events,
):
    """Attach cumulative realized P&L known before each trade."""

    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                realized_events.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    realized_events[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    realized_events[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                "open_date_time"
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                "realized_pnl_before_trade"
            ] = 0.0

            result_frames.append(
                current_group
            )

            continue

        group_events = (
            group_events.rename(
                columns={
                    "close_date_time":
                        (
                            "latest_realized_"
                            "close_before_trade"
                        )
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                group_events
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "latest_realized_"
                "close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        current_group[
            "realized_pnl_before_trade"
        ] = (
            current_group[
                "cumulative_realized_pnl"
            ]
            .fillna(0.0)
        )

        current_group = (
            current_group.drop(
                columns=[
                    "cumulative_realized_pnl"
                ],
                errors="ignore",
            )
        )

        result_frames.append(
            current_group
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )

## Position sizing

In [40]:
def attach_historical_position_size_features(
    target_table,
):
    """Attach historical median position-size features."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        medians = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            medians.append(
                np.median(
                    historical_amounts
                )
                if historical_amounts
                else np.nan
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = medians

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result[
            "amount"
        ]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    return result

In [41]:
def attach_historical_sizing_consistency_features(
    target_table,
):
    """Attach historical position-size coefficient of variation."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_cvs = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            past_mean = (
                np.mean(
                    historical_amounts
                )
                if past_count >= 1
                else np.nan
            )

            past_std = (
                np.std(
                    historical_amounts,
                    ddof=0,
                )
                if past_count >= 2
                else np.nan
            )

            past_cv = (
                past_std / past_mean
                if (
                    past_count >= 2
                    and past_mean > 0
                )
                else np.nan
            )

            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_cv"
        ] = (
            past_cvs
        )

        result_frames.append(
            current
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )

## Historical performance

In [42]:
def build_historical_performance_features(
    idea_table,
):
    """Build historical completed-idea performance features."""

    idea_performance = (
        idea_table[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
                "idea_end_time",
                "total_amount",
                "total_net_profit",
                "is_profitable_idea",
                "is_losing_idea",
                "is_breakeven_idea",
            ]
        ]
        .copy()
    )

    idea_performance[
        "idea_profit_per_lot"
    ] = (
        idea_performance[
            "total_net_profit"
        ]
        / idea_performance[
            "total_amount"
        ]
    )

    idea_performance[
        "_win_count"
    ] = (
        idea_performance[
            "is_profitable_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_loss_count"
    ] = (
        idea_performance[
            "is_losing_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_idea_count"
    ] = 1

    idea_performance[
        "_win_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .where(
            idea_performance[
                "is_profitable_idea"
            ],
            0.0,
        )
    )

    idea_performance[
        "_loss_abs_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .abs()
        .where(
            idea_performance[
                "is_losing_idea"
            ],
            0.0,
        )
    )

    events = (
        idea_performance
        .groupby(
            [
                "account_id",
                "idea_end_time",
            ],
            as_index=False,
        )
        .agg(
            completed_idea_count=(
                "_idea_count",
                "sum",
            ),
            completed_win_count=(
                "_win_count",
                "sum",
            ),
            completed_loss_count=(
                "_loss_count",
                "sum",
            ),
            completed_win_profit_sum=(
                "_win_profit",
                "sum",
            ),
            completed_loss_abs_profit_sum=(
                "_loss_abs_profit",
                "sum",
            ),
        )
        .sort_values(
            [
                "account_id",
                "idea_end_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    cumulative_mapping = {
        "past_completed_idea_count":
            "completed_idea_count",
        "past_winning_idea_count":
            "completed_win_count",
        "past_losing_idea_count":
            "completed_loss_count",
        "past_win_profit_sum":
            "completed_win_profit_sum",
        "past_loss_abs_profit_sum":
            (
                "completed_loss_"
                "abs_profit_sum"
            ),
    }

    for (
        output_column,
        source_column,
    ) in (
        cumulative_mapping.items()
    ):
        events[
            output_column
        ] = (
            events
            .groupby(
                "account_id"
            )[
                source_column
            ]
            .cumsum()
        )

    current_ideas = (
        idea_performance[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
            ]
        ]
        .sort_values(
            [
                "idea_start_time",
                "account_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    events = (
        events
        .sort_values(
            [
                "idea_end_time",
                "account_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    historical = pd.merge_asof(
        current_ideas,
        events[
            [
                "account_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on=(
            "idea_start_time"
        ),
        right_on=(
            "idea_end_time"
        ),
        by="account_id",
        direction="backward",
        allow_exact_matches=False,
    )

    historical[
        "past_idea_win_rate"
    ] = (
        historical[
            "past_winning_idea_count"
        ]
        / historical[
            "past_completed_idea_count"
        ]
    )

    historical[
        "past_mean_win_profit_per_lot"
    ] = (
        historical[
            "past_win_profit_sum"
        ]
        / historical[
            "past_winning_idea_count"
        ]
    )

    historical[
        "past_mean_loss_abs_profit_per_lot"
    ] = (
        historical[
            "past_loss_abs_profit_sum"
        ]
        / historical[
            "past_losing_idea_count"
        ]
    )

    historical[
        "past_payoff_ratio"
    ] = (
        historical[
            "past_mean_win_profit_per_lot"
        ]
        / historical[
            "past_mean_loss_abs_profit_per_lot"
        ]
    )

    return historical

## Model feature table

In [43]:
def build_model_features(
    stage1_tables,
):
    """Recreate the 22 pre-trade features used by the frozen model."""

    trades = (
        stage1_tables[
            "trades"
        ]
        .copy()
    )

    trades_with_ideas = (
        stage1_tables[
            "trades_with_ideas"
        ]
        .copy()
    )

    previous_completed = (
        stage1_tables[
            "trades_with_previous_completed"
        ]
        .copy()
    )

    idea_features = (
        stage1_tables[
            "idea_features"
        ]
        .copy()
    )

    trade_features = (
        trades_with_ideas
        .copy()
    )

    # --------------------------------------------------------
    # Loss response
    # --------------------------------------------------------

    previous_columns = [
        "trade_row_id",
        "previous_completed_close_date_time",
        "previous_completed_net_profit",
        "previous_completed_amount",
        "previous_completed_was_loss",
        "previous_completed_was_win",
        "reentry_gap_minutes",
    ]

    trade_features = (
        trade_features.merge(
            previous_completed[
                previous_columns
            ],
            on="trade_row_id",
            how="left",
            validate="one_to_one",
        )
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "amount"
        ]
        / trade_features[
            "previous_completed_amount"
        ]
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    trade_features[
        "post_loss_entry"
    ] = (
        trade_features[
            "previous_completed_was_loss"
        ]
        .fillna(False)
        .astype(bool)
    )

    trade_features[
        "post_loss_reentry_gap_minutes"
    ] = (
        trade_features[
            "reentry_gap_minutes"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    trade_features[
        "post_loss_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    # --------------------------------------------------------
    # Activity
    # --------------------------------------------------------

    trade_features = (
        attach_past_event_count(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            output_column=(
                "past_trade_open_count"
            ),
        )
    )

    trade_features[
        "first_trade_open_date_time"
    ] = (
        trade_features
        .groupby(
            [
                "account_id",
                "campaign_id",
            ]
        )[
            "open_date_time"
        ]
        .transform(
            "min"
        )
    )

    trade_features[
        "elapsed_active_hours"
    ] = (
        (
            trade_features[
                "open_date_time"
            ]
            - trade_features[
                "first_trade_open_date_time"
            ]
        )
        .dt.total_seconds()
        / 3600
    )

    valid_elapsed = (
        trade_features[
            "elapsed_active_hours"
        ]
        > 0
    )

    trade_features[
        "past_trades_opened_per_active_hour"
    ] = np.where(
        valid_elapsed,
        (
            trade_features[
                "past_trade_open_count"
            ]
            / trade_features[
                "elapsed_active_hours"
            ]
        ),
        np.nan,
    )

    trade_features = (
        attach_rolling_event_counts(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            window_minutes=[
                30,
            ],
            feature_prefix=(
                "trades_opened"
            ),
        )
    )

    # --------------------------------------------------------
    # Trade timing
    # --------------------------------------------------------

    trade_features = (
        attach_entry_spacing_features(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            tie_breaker_columns=[
                "trade_row_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_trade_open"
            ),
            past_median_gap_column=(
                "past_median_trade_open_gap_minutes"
            ),
            gap_ratio_column=(
                "trade_open_gap_to_past_median_ratio"
            ),
        )
    )

    idea_start_events = (
        idea_features[
            [
                "account_id",
                "campaign_id",
                "idea_id",
                "idea_start_time",
            ]
        ]
        .drop_duplicates(
            subset=[
                "idea_id"
            ]
        )
        .copy()
    )

    idea_features = (
        attach_entry_spacing_features(
            target_table=(
                idea_features
            ),
            event_table=(
                idea_start_events
            ),
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "idea_start_time"
            ),
            event_time_column=(
                "idea_start_time"
            ),
            tie_breaker_columns=[
                "idea_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_idea_start"
            ),
            past_median_gap_column=(
                "past_median_idea_start_gap_minutes"
            ),
            gap_ratio_column=(
                "idea_start_gap_to_past_median_ratio"
            ),
        )
    )

    trade_features = (
        trade_features.merge(
            idea_features[
                [
                    "idea_id",
                    "minutes_since_previous_idea_start",
                    "past_median_idea_start_gap_minutes",
                    "idea_start_gap_to_past_median_ratio",
                ]
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # --------------------------------------------------------
    # Challenge state
    # --------------------------------------------------------

    realized_events = (
        trades[
            [
                "account_id",
                "campaign_id",
                "close_date_time",
                "net_profit",
            ]
        ]
        .dropna(
            subset=[
                "close_date_time"
            ]
        )
        .groupby(
            [
                "account_id",
                "campaign_id",
                "close_date_time",
            ],
            as_index=False,
        )[
            "net_profit"
        ]
        .sum()
        .rename(
            columns={
                "net_profit":
                    "realized_pnl_at_close"
            }
        )
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "close_date_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    realized_events[
        "cumulative_realized_pnl"
    ] = (
        realized_events
        .groupby(
            [
                "campaign_id",
                "account_id",
            ]
        )[
            "realized_pnl_at_close"
        ]
        .cumsum()
    )

    trade_features = (
        attach_realized_challenge_state(
            target_table=(
                trade_features
            ),
            realized_events=(
                realized_events
            ),
        )
    )

    trade_features[
        "realized_distance_to_drawdown_limit"
    ] = (
        trade_features[
            "realized_pnl_before_trade"
        ]
        - REALIZED_DRAWDOWN_BOUNDARY
    )

    # --------------------------------------------------------
    # Position sizing
    # --------------------------------------------------------

    trade_features = (
        attach_historical_position_size_features(
            trade_features
        )
    )

    trade_features = (
        attach_historical_sizing_consistency_features(
            trade_features
        )
    )

    # --------------------------------------------------------
    # Historical idea performance
    # --------------------------------------------------------

    historical_idea_features = (
        build_historical_performance_features(
            idea_features
        )
    )

    historical_columns = [
        "idea_id",
        "past_completed_idea_count",
        "past_idea_win_rate",
        "past_mean_win_profit_per_lot",
        "past_mean_loss_abs_profit_per_lot",
        "past_payoff_ratio",
    ]

    trade_features = (
        trade_features.merge(
            historical_idea_features[
                historical_columns
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Replace invalid numerical values exactly as missing values.
    trade_features = (
        trade_features.replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    if (
        "reverse_profit"
        in trade_features.columns
        and "amount"
        in trade_features.columns
    ):
        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit"
            ]
            / trade_features[
                "amount"
            ]
        )

        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit_per_lot"
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
        )

    return (
        trade_features
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [ ]:
print("="*70)
print("DATA PREPARATION")
print("="*70)

stage1_tables = build_stage1_inference_tables(HISTORICAL_TRADES_DIR,UNSEEN_TRADES_DIR)

print("\nBuilding leakage-safe behavioural features...")

feature_table = build_model_features(stage1_tables)

print("Feature engineering completed.")
print("Feature rows:",f"{len(feature_table):,}")
print("Historical rows:",f"{int((~feature_table['_is_unseen']).sum()):,}")
print("Unseen rows:",f"{int(feature_table['_is_unseen'].sum()):,}")

DATA PREPARATION

Scanning historical trade directory:
  /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/User Trades
Found 20 historical files.
  [1/20] Processing: Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv
  [2/20] Processing: Campaign 34 Data 03 Mar 2026 XAUUSD only (1D).csv
  [3/20] Processing: Campaign 35 Data 06 Mar 2026 XAUUSD only (1D).csv
  [4/20] Processing: Campaign 36 Data 09 Mar 2026 XAUUSD only (1D).csv
  [5/20] Processing: Campaign 37 Data 13 Mar 2026 XAUUSD only (1D).csv
  [6/20] Processing: Campaign 38 Data 17 Mar 2026 XAUUSD only (1D).csv
  [7/20] Processing: Campaign 39 Data 20 Mar 2026 XAUUSD only (1D).csv
  [8/20] Processing: Campaign 40 Data 24 Mar 2026 XAUUSD only (1D).csv
  [9/20] Processing: Campaign 41 Data 27 Mar 2026 XAUUSD only (1D).csv
  [10/20] Processing: Campaign 42 Data 31 Mar 2026 XAUUSD only (1D).csv
  [11/20] Processing: Campaign 43 Data 6 Apr 2026 XAUUSD only (1D).csv
  [12/20] Processing: Campaign 44 D

# Model

## Frozen specification

In [ ]:
def build_frozen_model():
    return LGBMRegressor(**LIGHTGBM_PARAMETERS)

def build_model_metadata(training_campaigns):
    return {
        "experiment":"exp10b",
        "description":"reduced_feature_regression",
        "model_type":"lightgbm",
        "target":"reverse_profit_per_lot",
        "score_column":"predicted_reverse_profit_per_lot",
        "feature_columns":FEATURE_COLUMNS,
        "n_features":len(FEATURE_COLUMNS),
        "training_campaigns":[int(x) for x in sorted(training_campaigns)],
        "selection_quantile":0.90,
        "decision_threshold":DECISION_THRESHOLD,
        "decision_rule":"fade if predicted_reverse_profit_per_lot >= decision_threshold",
        "random_seed":RANDOM_SEED,
        "lightgbm_parameters":LIGHTGBM_PARAMETERS,
        "history_key":"account_id",
        "bootstrap_cluster_key":"account_id",
        "stand_down_filter_enabled":STAND_DOWN_FILTER_ENABLED
    }

def save_frozen_model(model, metadata):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, MODEL_PATH)
    MODEL_METADATA_PATH.write_text(json.dumps(metadata, indent=4),encoding="utf-8")
    
    print("Frozen model saved:",MODEL_PATH)

def validate_model_metadata(metadata):
    if list(metadata.get("feature_columns",[]))!=list(FEATURE_COLUMNS):
        raise RuntimeError("Frozen model feature list does not match expected 22 features.")
    
    if not np.isclose(float(metadata.get("decision_threshold")),DECISION_THRESHOLD):
        raise RuntimeError("Frozen decision threshold does not match expected value.")
    
    if metadata.get("model_type")!="lightgbm":
        raise RuntimeError("Frozen model is not LightGBM.")

## Load or rebuild

In [ ]:
def load_or_rebuild_model(feature_table):
    if MODEL_PATH.exists() and MODEL_METADATA_PATH.exists():
        print("Frozen model found. Skipping training.")
        
        model = joblib.load(MODEL_PATH)
        metadata = json.loads(MODEL_METADATA_PATH.read_text(encoding="utf-8"))
        
        validate_model_metadata(metadata)
        
        print("Frozen model loaded successfully.")
        
        return model,metadata,"loaded"
    
    print("Frozen model not found or incomplete.")
    print("Rebuilding fixed Experiment 10B model...")
    
    training = feature_table.loc[(~feature_table['_is_unseen']) & feature_table['campaign_id'].between(TRAINING_CAMPAIGN_MIN,TRAINING_CAMPAIGN_MAX)].copy()
    missing = sorted(set(FEATURE_COLUMNS)-set(training.columns))
    
    if missing: 
        raise RuntimeError(f"Missing model features: {missing}")
    
    if 'reverse_profit_per_lot' not in training.columns: 
        raise RuntimeError('Training target reverse_profit_per_lot is unavailable.')
    
    training=training.loc[training['reverse_profit_per_lot'].notna()].reset_index(drop=True)
    
    if training.empty: raise RuntimeError('No valid Campaign 33-52 training rows were found.')
    
    campaigns = sorted(training['campaign_id'].dropna().unique())
    
    print("Training campaigns:",campaigns)
    print("Training rows:",f"{len(training):,}")
    
    model = build_frozen_model()
    
    print("Fitting frozen LightGBM...")
    
    model.fit(training[FEATURE_COLUMNS],training['reverse_profit_per_lot'])
    metadata = build_model_metadata(campaigns)
    save_frozen_model(model,metadata)
    
    print("Frozen model rebuild completed.")
    
    return model,metadata,"rebuilt"

model,model_metadata,model_source = load_or_rebuild_model(feature_table)

print("Model source:",model_source)

Frozen model found. Skipping training.
Frozen model loaded successfully.
Model source: loaded


# Inference

## Stand-down filter

In [47]:
def get_stand_down_mask(feature_table):
    if not STAND_DOWN_FILTER_ENABLED:
        return pd.Series(False,index=feature_table.index,dtype=bool)
    raise NotImplementedError("Stand-down filter is enabled but not yet defined. Update this function after the losing-band rule is clarified.")

print("Stand-down filter enabled:",STAND_DOWN_FILTER_ENABLED)

Stand-down filter enabled: False


## Score unseen trades

In [ ]:
print("="*70)
print("INFERENCE")
print("="*70)

banned = {"profit","net_profit","reverse_profit","reverse_profit_per_lot","sl_price","tp_price"}
leaking = set(FEATURE_COLUMNS) & banned

if leaking: 
    raise RuntimeError(f"Leakage detected: {sorted(leaking)}")

missing = sorted(set(FEATURE_COLUMNS) - set(feature_table.columns))

if missing: 
    raise RuntimeError(f"Required model features are missing: {missing}")

unseen_features = feature_table.loc[feature_table['_is_unseen']].copy().reset_index(drop=True)

if unseen_features.empty: 
    raise RuntimeError('No unseen trades were found.')

print("Scoring unseen trades:",f"{len(unseen_features):,}")

unseen_features['predicted_reverse_profit_per_lot'] = model.predict(unseen_features[FEATURE_COLUMNS])
unseen_features['decision_threshold'] = DECISION_THRESHOLD
unseen_features['raw_fade_decision'] = unseen_features['predicted_reverse_profit_per_lot']>=DECISION_THRESHOLD
unseen_features['stand_down'] = get_stand_down_mask(unseen_features).astype(bool)
unseen_features['fade_decision'] = (unseen_features['raw_fade_decision'] & ~unseen_features['stand_down']).astype(int)

cols = ['trade_row_id',
    'campaign_id',
    'account_id',
    'source_file',
    'source_row_number',
    'open_date_time',
    'predicted_reverse_profit_per_lot',
    'decision_threshold',
    'raw_fade_decision',
    'stand_down',
    'fade_decision'
    ]

unseen_trade_decisions = unseen_features[cols].sort_values(['campaign_id','account_id','open_date_time','trade_row_id']).reset_index(drop=True)
n_unseen = len(unseen_trade_decisions); n_faded=int(unseen_trade_decisions['fade_decision'].sum()); coverage=n_faded/n_unseen if n_unseen else np.nan

print("Inference completed.")
print("Trades evaluated:",f"{n_unseen:,}")
print("Trades faded:",f"{n_faded:,}")
print("Coverage:",f"{coverage:.2%}")

INFERENCE
Scoring unseen trades: 52,853
Inference completed.
Trades evaluated: 52,853
Trades faded: 8,227
Coverage: 15.57%


# Evaluation

## Metrics

In [ ]:
def has_complete_realized_outcomes(df):
    req = {'reverse_profit','amount'}
    
    return req.issubset(df.columns) and not df['reverse_profit'].isna().any() and not df['amount'].isna().any() and not (df['amount']<=0).any()

def calculate_lot_weighted_edge(df):
    if df.empty: 
        return np.nan
    
    lots = df['amount'].sum()
    
    return df['reverse_profit'].sum() / lots if np.isfinite(lots) and lots>0 else np.nan

def evaluate_by_campaign(df):
    rows = []
    
    for cid,g in df.groupby('campaign_id',sort=True,dropna=False):
        faded=g.loc[g['fade_decision']==1]
        rows.append({
            'campaign_id':cid,
            'trades_evaluated':len(g),
            'trades_faded':len(faded),
            'coverage':len(faded)/len(g) if len(g) else np.nan,
            'lot_weighted_rp_per_lot':calculate_lot_weighted_edge(faded)
        })
    
    out=pd.DataFrame(rows)
    out['is_positive_campaign'] = out['lot_weighted_rp_per_lot'].gt(0).fillna(False)
    
    return out

def account_cluster_bootstrap(df,n_bootstrap=N_BOOTSTRAP,random_seed=RANDOM_SEED):
    faded=df.loc[df['fade_decision']==1].copy()
    
    if faded.empty: 
        return pd.DataFrame(columns=['bootstrap_iteration','lot_weighted_rp_per_lot'])
    
    accounts=faded['account_id'].dropna().unique()
    
    if len(accounts)==0: 
        return pd.DataFrame(columns=['bootstrap_iteration','lot_weighted_rp_per_lot'])
    
    groups={a:faded.loc[faded['account_id']==a].copy() for a in accounts}
    rng=np.random.default_rng(random_seed); rows=[]; step=max(n_bootstrap//5,1)
    
    print("Running account-cluster bootstrap:",f"{n_bootstrap:,} iterations")
    
    for i in range(n_bootstrap):
        sampled=rng.choice(accounts,size=len(accounts),replace=True)
        sample=pd.concat([groups[a] for a in sampled],ignore_index=True)
        rows.append({'bootstrap_iteration':i+1,'lot_weighted_rp_per_lot':calculate_lot_weighted_edge(sample)})
        if (i+1)%step==0: print("  Bootstrap progress:",f"{i+1:,}/{n_bootstrap:,}")

    return pd.DataFrame(rows)

def confidence_interval(values,confidence_level=CONFIDENCE_LEVEL):
    x = pd.Series(values).dropna().to_numpy()
    if len(x)==0: 
        return np.nan,np.nan
    
    alpha=1-confidence_level
    
    return float(np.quantile(x,alpha/2)),float(np.quantile(x,1-alpha/2))

## Run evaluation

In [ ]:
print("="*70)
print("EVALUATION")
print("="*70)

evaluation_available = has_complete_realized_outcomes(unseen_features)
evaluation_summary = campaign_performance=bootstrap_samples=None

if not evaluation_available:
    print("Realized outcomes are not fully available.")
    print("Evaluation skipped. Inference results remain available.")
else:
    evaluation_data = unseen_features[['trade_row_id','campaign_id','account_id','amount','reverse_profit','fade_decision']].copy()
    faded = evaluation_data.loc[evaluation_data['fade_decision']==1].copy()
    trades_evaluated = len(evaluation_data); trades_faded=len(faded); coverage=trades_faded/trades_evaluated if trades_evaluated else np.nan
    edge = calculate_lot_weighted_edge(faded)
    
    print("Calculating campaign-level performance...")
    
    campaign_performance = evaluate_by_campaign(evaluation_data)
    positive_campaigns = int(campaign_performance['is_positive_campaign'].sum()); total_campaigns=len(campaign_performance); positive_share=positive_campaigns/total_campaigns if total_campaigns else np.nan
    bootstrap_samples = account_cluster_bootstrap(evaluation_data)
    ci_lower,ci_upper = confidence_interval(bootstrap_samples['lot_weighted_rp_per_lot'])
    evaluation_summary = pd.DataFrame([{'trades_evaluated':trades_evaluated,'trades_faded':trades_faded,'coverage':coverage,'lot_weighted_rp_per_lot':edge,'ci_lower':ci_lower,'ci_upper':ci_upper,'positive_campaigns':positive_campaigns,'total_campaigns':total_campaigns,'positive_campaign_share':positive_share}])
    
    print("\nEvaluation completed.")
    print("Trades evaluated:",f"{trades_evaluated:,}")
    print("Trades faded:",f"{trades_faded:,}")
    print("Coverage:",f"{coverage:.2%}")
    print("Lot-weighted RP/lot:",f"{edge:.6f}")
    print("95% CI:",f"[{ci_lower:.6f}, {ci_upper:.6f}]")
    print("Positive campaigns:",f"{positive_campaigns}/{total_campaigns}")
    print("Positive campaign share:",f"{positive_share:.2%}")
    display(evaluation_summary)
    display(campaign_performance)

EVALUATION
Calculating campaign-level performance...
Running account-cluster bootstrap: 1,000 iterations
  Bootstrap progress: 200/1,000
  Bootstrap progress: 400/1,000
  Bootstrap progress: 600/1,000
  Bootstrap progress: 800/1,000
  Bootstrap progress: 1,000/1,000

Evaluation completed.
Trades evaluated: 52,853
Trades faded: 8,227
Coverage: 15.57%
Lot-weighted RP/lot: 6.915446
95% CI: [-5.795576, 20.769467]
Positive campaigns: 16/30
Positive campaign share: 53.33%


,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share
0,52853,8227,0.155658,6.915446,-5.795576,20.769467,16,30,0.533333


,campaign_id,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,is_positive_campaign
0,53,1063,85,0.079962,82.486034,True
1,54,1511,99,0.065520,55.775588,True
2,55,1222,92,0.075286,-82.340029,False
3,56,1732,129,0.074480,-112.416903,False
4,57,1256,96,0.076433,22.479167,True
5,58,1263,88,0.069675,-79.661945,False
6,59,1490,141,0.094631,190.961744,True
7,60,1575,144,0.091429,-31.418249,False
8,61,1573,156,0.099174,16.466278,True
9,62,1602,157,0.098002,36.387384,True


# Outputs

## Save results

In [51]:
print("="*70)
print("SAVING OUTPUTS")
print("="*70)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
unseen_trade_decisions.to_csv(DECISIONS_OUTPUT_PATH,index=False)
print("Saved trade decisions:",DECISIONS_OUTPUT_PATH)
if evaluation_summary is not None:
    evaluation_summary.to_csv(EVALUATION_SUMMARY_PATH,index=False)
    campaign_performance.to_csv(CAMPAIGN_PERFORMANCE_PATH,index=False)
    bootstrap_samples.to_csv(BOOTSTRAP_SAMPLES_PATH,index=False)
    print("Saved evaluation summary:",EVALUATION_SUMMARY_PATH)
    print("Saved campaign performance:",CAMPAIGN_PERFORMANCE_PATH)
    print("Saved bootstrap samples:",BOOTSTRAP_SAMPLES_PATH)
else:
    print("No evaluation files saved because realized outcomes were unavailable.")
print("Pipeline complete.")

SAVING OUTPUTS
Saved trade decisions: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/inference/unseen_trade_decisions.csv
Saved evaluation summary: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/inference/evaluation_summary.csv
Saved campaign performance: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/inference/campaign_performance.csv
Saved bootstrap samples: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/inference/bootstrap_samples.csv
Pipeline complete.
